# 🏆 AlphaKhulnasoft — Competitive Programming: Solving Novel Problems

This notebook demonstrates the `alphakhulnasoft.contests` package: solving *unseen* competitive-programming problems **without copying a reference solution**. It runs entirely on the in-repo fixture (`data/tiny.jsonl`) and a **fake LLM**, so no API key is required.

Key idea: the generator only sees the problem statement + sample I/O; reference solutions are quarantined behind `get_references` and used solely by the verifier to measure output-overlap / memorization. We report both `pass@k` and `novel_pass@k` — the milestone metric.

In [ ]:
import json
from pathlib import Path

import alphakhulnasoft.contests as cc
from alphakhulnasoft.contests.harness import grade_solution
from alphakhulnasoft.contests.loader import load_local, get_references

## 1. Load the fixture
Three a+b problems from codeforces / atcoder / codechef, each with visible + hidden tests and Python + C++ reference solutions.

In [ ]:
FIXTURE = str(Path(cc.__file__).parent / "data" / "tiny.jsonl")
problems = load_local(FIXTURE)
print(f"{len(problems)} problems:", [p.problem_id for p in problems])
problems[0]

## 2. Grade a candidate (sandbox)
The harness runs a candidate in a sandbox and returns per-test `PASS/WA/RE/CE/TLE/MLE`. References are quarantined — `get_references` is verifier-only and must never be called by the generator.

In [ ]:
p = problems[0]
correct = "import sys\na,b=map(int,sys.stdin.read().split())\nprint(a+b)"
wrong   = "import sys\na,b=map(int,sys.stdin.read().split())\nprint(a-b)"
print("correct:", grade_solution(p, correct, "py").all_passed())
print("wrong  :", grade_solution(p, wrong, "py").all_passed())
# References are NOT given to the generator; only the verifier may read them:
print("references (py):", [r.status for r in get_references(p, "py")])

## 3. Generate + repair with a fake LLM (no key needed)
`ContestAgent` runs the Analyze → Generate → Verify → Repair loop. We substitute a fake LLM that returns a correct solution, plus a 'historian' probe that would regurgitate a reference.

In [ ]:
class FakeLLM:
    def complete(self, prompt, system_prompt=None):
        if system_prompt and "historian" in system_prompt:
            # memorization probe: would recall the canonical reference verbatim
            return "import sys\na,b=map(int,sys.stdin.read().split())\nprint(a+b)"
        return "import sys\nx,y=map(int,sys.stdin.read().split())\nprint(x+y)"
    def extract_code(self, text):
        return text.strip()

agent = cc.ContestAgent(llm=FakeLLM())
cand = agent.solve(p, "py", n_samples=4)
print("status:", cand.status, "| visible_pass:", cand.grade.all_passed())

## 4. Benchmark: pass@k vs novel_pass@k

In [ ]:
report = cc.run_benchmark(problems, agent, "py", n_samples=6, k=2)
print(json.dumps(report.as_dict(), indent=2))

## 5. The novelty guard
`novel_pass@k` counts a problem as *solved* only when the chosen solution is **not** a near-duplicate of any reference (token similarity `< 0.7`). The fake above is token-similar to the reference, so its `novel_pass@k` is 0 even though `pass@k` is 1.0 — exactly the 'copying' failure mode the milestone guards against.

Run on the real dataset with an API key:
```bash
uv run python -m alphakhulnasoft.contests bench --hf-dataset code_contests --split valid --limit 200 --n-samples 10 --k 5
```